# Task 2 — Classificatore Manuale: Decision Tree

In questo notebook viene progettato e implementato **manualmente** un secondo classificatore
sul dataset `manuale.csv`: un **albero di decisione (Decision Tree)**.

Mentre nel notebook precedente è stato sviluppato un classificatore *Naive Bayes* (di tipo probabilistico),
qui si adotta un modello *basato su regole*. Questo permette, in fase di esame, di **confrontare due
famiglie di classificatori diverse** sullo stesso insieme di dati.

La procedura seguita è la stessa del notebook 02:
1. caricamento di `manuale.csv` e separazione feature/target;
2. richiamo teorico del modello (entropia e information gain);
3. costruzione manuale dell'albero, passo dopo passo, scegliendo gli split in base all'information gain;
4. implementazione in Python della funzione di predizione;
5. valutazione delle prestazioni (accuracy, confusion matrix, precision, recall, F1);
6. discussione critica dei risultati.

In [1]:
import pandas as pd
import numpy as np

## 1. Caricamento del dataset `manuale.csv`

Si carica lo stesso file `manuale.csv` utilizzato per il Naive Bayes: 14 campioni bilanciati
(7 di classe 0 e 7 di classe 1) rispetto alla variabile target `Diabetes_binary`.
Utilizzare lo stesso file consente un confronto diretto tra i due classificatori manuali.

In [2]:
# Caricamento del file manuale.csv

manual_df = pd.read_csv("../data/manuale.csv")

manual_df.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,0,1,21,0,0,0,1,0,1,...,0,3,3,2,0,0,2,6,8,0
1,0,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,13,5,6,0
2,1,1,1,33,1,0,0,1,0,1,...,0,3,0,0,0,1,8,4,8,1
3,0,0,1,23,1,0,0,0,1,1,...,0,3,0,0,0,0,5,3,5,0
4,0,1,1,37,0,0,0,0,1,1,...,0,4,20,30,1,0,10,2,2,1


## 2. Separazione tra feature e target

- `X_manual`: contiene tutte le feature esplicative.
- `y_manual`: contiene la variabile target `Diabetes_binary`.

In [3]:
# Separiamo le feature dalla variabile target

X_manual = manual_df.drop(columns=["Diabetes_binary"])
y_manual = manual_df["Diabetes_binary"]

# Controlliamo le dimensioni
print("Shape X_manual:", X_manual.shape)
print("Shape y_manual:", y_manual.shape)

# Verifichiamo la distribuzione della classe
y_manual.value_counts()

Shape X_manual: (14, 21)
Shape y_manual: (14,)


Diabetes_binary
0    7
1    7
Name: count, dtype: int64

## 3. Richiamo teorico: Albero di Decisione (Decision Tree)

Un **albero di decisione** è un classificatore che suddivide ricorsivamente i dati in
sottoinsiemi sempre più "puri" rispetto alla variabile target, applicando ad ogni nodo
un test su una feature.

La classificazione di una nuova osservazione avviene percorrendo l'albero dalla **radice**
fino a una **foglia**, seguendo ad ogni nodo il ramo corrispondente al valore della feature testata.
La classe assegnata è quella prevalente nella foglia raggiunta.

### Criterio di split: Entropia e Information Gain

Per decidere **quale feature** usare in ciascun nodo si utilizza il concetto di **entropia**,
che misura il grado di "disordine" (impurità) di un insieme rispetto alla classe.

Per un insieme S con due classi:

$$Entropia(S) = - \sum_{i} p_i \, \log_2(p_i)$$

dove $p_i$ è la proporzione di campioni della classe $i$.

- Se l'insieme contiene **una sola classe** (insieme puro), l'entropia vale **0**.
- Se le due classi sono **perfettamente bilanciate** (50%/50%), l'entropia è **massima e vale 1**.

Quando si effettua uno split su una feature A, si calcola l'**Information Gain**, cioè la riduzione
di entropia ottenuta:

$$IG(S, A) = Entropia(S) - \sum_{v \in valori(A)} \frac{|S_v|}{|S|} \, Entropia(S_v)$$

Ad ogni nodo si sceglie la feature che **massimizza l'Information Gain**: è quella che separa
meglio le due classi.

### Strategia adottata

Poiché alcune feature numeriche hanno troppi valori distinti per soli 14 campioni, si riutilizza
la **discretizzazione del BMI** già introdotta nel notebook precedente (Normal / Overweight / Obese),
in modo da rendere i conteggi più stabili e l'albero più interpretabile.

## 4. Calcolo dell'entropia del nodo radice

Prima di costruire l'albero, si calcola l'entropia dell'intero dataset rispetto al target.
Essendo il dataset perfettamente bilanciato (7 contro 7), ci si aspetta un'entropia pari a 1.

In [4]:
# Funzione per il calcolo dell'entropia

def entropy(y):                                               #riceve la colonna target
    if len(y) == 0:                                           #if messo per prevenire il crash, se l'insieme è vuoto, nessun campione
        return 0
    proportions = y.value_counts(normalize=True)              #normalize messo per far restituire 0.5 e 0.5, invece dei conteggi assoluti 7 e 7
    return -(proportions * np.log2(proportions)).sum()        #applicazione formula

entropia_radice = entropy(manual_df["Diabetes_binary"])
print(f"Entropia del nodo radice: {entropia_radice:.4f}")

Entropia del nodo radice: 1.0000


### Risultato

L'entropia del nodo radice è pari a **1.0**, come previsto: il dataset di partenza è
perfettamente bilanciato tra le due classi e rappresenta quindi la situazione di massima
incertezza. L'obiettivo dell'albero è ridurre progressivamente questa entropia.

## 5. Discretizzazione del BMI

Come nel notebook del Naive Bayes, la variabile `BMI` viene discretizzata in tre categorie.
Il BMI grezzo assumerebbe troppi valori distinti su soli 14 campioni, rendendo gli split
poco significativi.

- BMI < 25         → Normal
- 25 ≤ BMI < 30    → Overweight
- BMI ≥ 30         → Obese

In [5]:
# Discretizzazione del BMI

manual_df["BMI_Category"] = pd.cut(                 #metodo che prende una colonna di numeri e la trasforma
    manual_df["BMI"],                               #in una colonna categoriale
    bins=[0, 25, 30, 100],                          #4 numeri, 3 intervalli
    labels=["Normal", "Overweight", "Obese"],
    right=False                                     #bordo destro escluso, quindi ad esempio
)                                                   #il 25 è incluso in overweight, invece di normal

manual_df[["BMI", "BMI_Category"]]

,BMI,BMI_Category
0,21,Normal
1,27,Overweight
2,33,Obese
3,23,Normal
4,37,Obese
5,25,Overweight
6,26,Overweight
7,29,Overweight
8,24,Normal
9,29,Overweight


## 6. Funzione per il calcolo dell'Information Gain

Si definisce una funzione che, data una feature, calcola l'Information Gain rispetto al target.
Questa funzione verrà usata ad ogni nodo per scegliere la feature di split migliore.
Mentre la funzione entropy misura il disordine di un gruppo, l'information gain misura quanto disordine elimini
dividendo i dati con una certa feature.


In [6]:
# Funzione per il calcolo dell'Information Gain

def information_gain(df, feature, target="Diabetes_binary"):
    entropia_padre = entropy(df[target])
    n = len(df)                                                       #per normalizzare, infatti si trova al denominatore di peso
    entropia_figli = 0

    for valore, sottogruppo in df.groupby(feature, observed=True):    #spacca il dataframe in altri piccoli raggruppandoli per feature,
        peso = len(sottogruppo) / n                                   #principalmente fatto per quando arriva la colonna BMI_Category.
        entropia_figli += peso * entropy(sottogruppo[target])         #Quindi se groupby viene messo in un ciclo for, ci da due valori:
                                                                      #il nome del sottogruppo(valore) e il "mini dataframe" (sottogruppo)
    return entropia_padre - entropia_figli

# ---------------------------------------------------------------------------
# VERSIONE ALTERNATIVA "PYTHONICA" (vettoriale, senza ciclo for esplicito)
# ---------------------------------------------------------------------------
# La stessa identica formula puo' essere scritta in forma vettoriale,
# lasciando che pandas/NumPy eseguano l'iterazione internamente:
#
# def information_gain_vectorized(df, feature, target="Diabetes_binary"):
#     entropia_padre = entropy(df[target])
#     n = len(df)
#     gruppi = df.groupby(feature, observed=True)[target]
#     entropie = gruppi.apply(entropy)         # entropia di ciascun gruppo
#     pesi = gruppi.size() / n                  # peso (frazione) di ciascun gruppo
#     entropia_figli = (pesi * entropie).sum()  # media pesata = prodotto scalare
#     return entropia_padre - entropia_figli
#
# Le due versioni producono risultati NUMERICAMENTE IDENTICI (verificato su
# tutte le feature del dataset).
#
# PERCHE' ABBIAMO SCELTO LA VERSIONE CON IL CICLO FOR:
#   1. Leggibilita' / spiegabilita': il ciclo ricalca passo-passo la formula
#      teorica IG = Entropia(padre) - Sum_v [ |S_v|/|S| * Entropia(S_v) ].
#      Si vede esplicitamente "per ogni ramo: calcola il peso, calcola
#      l'entropia, accumula", rendendo immediato il legame codice-teoria.
#   2. Il progetto richiede che ogni membro sappia spiegare in dettaglio il
#      codice: il ciclo esplicito e' piu' trasparente della versione compatta.
#   3. La velocita' superiore della versione vettoriale e' rilevante solo su
#      dataset molto grandi; qui (14 campioni e piccoli sottoinsiemi) la
#      differenza e' nell'ordine dei microsecondi, quindi trascurabile.
# ---------------------------------------------------------------------------

## 7. Scelta dello split alla radice

Si calcola l'Information Gain di tutte le feature candidate per individuare quella che, alla radice,
separa meglio le due classi.

In [7]:
# Information Gain delle feature candidate sulla radice

candidate_features = ["HighBP", "HighChol", "BMI_Category", "GenHlth", "PhysActivity"]

ig_results = {}
for f in candidate_features:
    ig_results[f] = information_gain(manual_df, f)

ig_df = pd.DataFrame(
    sorted(ig_results.items(), key=lambda x: x[1], reverse=True),
    columns=["Feature", "Information Gain"]
)
ig_df

,Feature,Information Gain
0,BMI_Category,0.313597
1,HighChol,0.257831
2,GenHlth,0.224661
3,PhysActivity,0.016112
4,HighBP,0.014772


### Osservazioni

La feature con Information Gain più elevato è **BMI_Category** (IG ≈ 0.314).
Viene quindi scelta come **nodo radice** dell'albero.

Si analizza ora come si distribuiscono le classi nei tre rami generati da questo split.

In [8]:
# Distribuzione delle classi nei rami di BMI_Category

pd.crosstab(manual_df["BMI_Category"], manual_df["Diabetes_binary"])

Diabetes_binary,0,1
BMI_Category,,
Normal,4,1
Overweight,3,3
Obese,0,3


### Analisi dei tre rami

Dallo split sulla radice si osserva:

- **BMI = Obese** → tutti e 3 i campioni appartengono alla classe 1 (diabetici).
  Il nodo è **puro** (entropia = 0): diventa una **foglia** che predice la classe **1**.
- **BMI = Normal** → 4 campioni di classe 0 e 1 di classe 1. Il nodo è **impuro** e va suddiviso ulteriormente.
- **BMI = Overweight** → 3 campioni di classe 0 e 3 di classe 1 (entropia = 1, massima impurità).
  Anche questo nodo va suddiviso ulteriormente.

Si procede quindi a espandere i due rami impuri (`Normal` e `Overweight`).

## 8. Espansione del ramo BMI = Normal

Si isolano i campioni con `BMI_Category = Normal` e si cerca la feature con Information Gain
maggiore per separarli.

In [9]:
# Sottoinsieme dei campioni Normal

normal_subset = manual_df[manual_df["BMI_Category"] == "Normal"]

normal_subset[["HighBP", "HighChol", "GenHlth", "PhysActivity", "Diabetes_binary"]]

,HighBP,HighChol,GenHlth,PhysActivity,Diabetes_binary
0,1,0,3,1,0
3,0,0,3,0,0
8,1,0,1,1,0
11,0,0,4,0,0
12,0,0,3,1,1


In [10]:
# Information Gain nel ramo Normal

normal_features = ["HighBP", "HighChol", "GenHlth", "PhysActivity"]

ig_values = list(map(lambda f: information_gain(normal_subset, f), normal_features))  #f assume i valori di normal_features
#map applica la funzione ad ogni elemento della lista f

ig_normal = pd.DataFrame({
    "Feature": normal_features,
    "Information Gain": ig_values
}).sort_values("Information Gain", ascending=False).reset_index(drop=True)

ig_normal

,Feature,Information Gain
0,HighBP,0.170951
1,GenHlth,0.170951
2,PhysActivity,0.170951
3,HighChol,0.000000


### Scelta dello split nel ramo Normal

Nel ramo `Normal`, la feature `HighBP` offre uno dei valori più alti di Information Gain ed è
una variabile clinicamente significativa (pressione alta). Viene scelta come split.

Analizzando i due rami:

- **HighBP = 0** → 2 campioni di classe 0 e 1 di classe 1 → maggioranza **classe 0**
- **HighBP = 1** → 2 campioni di classe 0 e 0 di classe 1 → foglia pura **classe 0**

Entrambi i rami portano a predire la **classe 0**. Il campione di classe 1 con BMI Normal
(riga 12) rappresenta un caso difficile che l'albero non riesce a separare correttamente con
queste feature: rimarrà un **errore residuo** (falso negativo).

In [11]:
# Distribuzione classi nel ramo Normal suddiviso per HighBP

pd.crosstab(normal_subset["HighBP"], normal_subset["Diabetes_binary"])

Diabetes_binary,0,1
HighBP,,
0,2,1
1,2,0


## 9. Espansione del ramo BMI = Overweight

Si ripete la stessa procedura per i campioni con `BMI_Category = Overweight`, il ramo più
impuro (entropia = 1, ovvero 3 campioni di classe 0 e 3 di classe 1).

In [12]:
# Sottoinsieme dei campioni Overweight

overweight_subset = manual_df[manual_df["BMI_Category"] == "Overweight"]

overweight_subset[["HighBP", "HighChol", "GenHlth", "PhysActivity", "Diabetes_binary"]]

,HighBP,HighChol,GenHlth,PhysActivity,Diabetes_binary
1,0,0,2,1,0
5,1,1,3,0,0
6,1,0,4,0,1
7,0,1,2,1,1
9,1,1,2,1,1
10,0,0,1,1,0


In [13]:
# Information Gain nel ramo Overweight (stessa logica del ramo Normal: map + lambda)

overweight_features = ["HighBP", "HighChol", "GenHlth", "PhysActivity"]

ig_values = list(map(lambda f: information_gain(overweight_subset, f), overweight_features))
#f assume i valori di overweight_features, map applica la funzione ad ogni elemento

ig_overweight = pd.DataFrame({
    "Feature": overweight_features,
    "Information Gain": ig_values
}).sort_values("Information Gain", ascending=False).reset_index(drop=True)

ig_overweight

,Feature,Information Gain
0,GenHlth,0.540852
1,HighBP,0.081704
2,HighChol,0.081704
3,PhysActivity,0.000000


### Scelta dello split nel ramo Overweight

A differenza del ramo Normal (dove tre feature avevano lo stesso Information Gain), qui la feature
**GenHlth** (stato di salute generale percepito, da 1 = ottimo a 5 = pessimo) ha un Information Gain
**nettamente superiore** a tutte le altre (≈ 0.541 contro ≈ 0.082). Viene quindi scelta come
feature di split per questo ramo, in coerenza con il criterio di massimizzazione dell'Information Gain
che guida la costruzione dell'albero.

Si analizza ora come si distribuiscono le classi rispetto ai valori di GenHlth in questo ramo.

In [14]:
# Distribuzione classi nel ramo Overweight suddiviso per GenHlth

pd.crosstab(overweight_subset["GenHlth"], overweight_subset["Diabetes_binary"])

Diabetes_binary,0,1
GenHlth,,
1,1,0
2,1,2
3,1,0
4,0,1


### Analisi dei valori di GenHlth nel ramo Overweight

Nel ramo Overweight la feature GenHlth assume quattro valori (1, 2, 3, 4), con la seguente
distribuzione delle classi:

| GenHlth | Classe 0 | Classe 1 | Classe assegnata (maggioranza) |
|---------|----------|----------|--------------------------------|
| 1       | 1        | 0        | **0** |
| 2       | 1        | 2        | **1** |
| 3       | 1        | 0        | **0** |
| 4       | 0        | 1        | **1** |

Ad ogni valore di GenHlth viene assegnata la classe prevalente. Si ottiene quindi la regola:

- GenHlth ∈ {1, 3} → **non diabetico (0)**
- GenHlth ∈ {2, 4} → **diabetico (1)**

**Osservazione critica:** alcune foglie contengono un solo campione (GenHlth = 1, 3, 4). Su un dataset
di soli 14 elementi questo è inevitabile, ma evidenzia un rischio concreto di **overfitting**: l'albero
sta in parte "memorizzando" i singoli campioni piuttosto che apprendere un pattern generale. Questo limite
verrà superato nei Task 4 e 5, dove i modelli saranno valutati su decine di migliaia di osservazioni.

## 10. Struttura finale dell'albero di decisione

Combinando tutti gli split scelti in base all'Information Gain, l'albero costruito manualmente
ha la seguente struttura:

```
                          [ BMI_Category ? ]
                         /        |          \
                   Normal     Overweight      Obese
                     |            |             |
                [ HighBP ? ]  [ GenHlth ? ]   CLASSE 1
                /        \      /       \
            =1 /       =0 \  {1,3}/   {2,4}\
          CLASSE 0   CLASSE 0  CLASSE 0   CLASSE 1
```

In forma di regole (leggibili e interpretabili):

1. **SE** BMI è *Obese* → **diabetico (1)**
2. **SE** BMI è *Overweight* **E** GenHlth ∈ {2, 4} → **diabetico (1)**
3. **SE** BMI è *Overweight* **E** GenHlth ∈ {1, 3} → **non diabetico (0)**
4. **SE** BMI è *Normal* → **non diabetico (0)** (indipendentemente da HighBP, poiché entrambi i rami concordano)

Rispetto a una prima versione dell'albero che utilizzava il colesterolo (HighChol) per il ramo Overweight,
questa struttura adotta GenHlth perché presenta un Information Gain molto più elevato su quel sotto-insieme,
risultando quindi una scelta più corretta dal punto di vista dell'algoritmo. Come si vedrà nella valutazione,
questa scelta migliora anche le prestazioni complessive del classificatore.

## 11. Implementazione del classificatore in Python

L'albero viene tradotto in una funzione Python che, data un'osservazione, percorre i nodi
e restituisce la classe predetta.

In [15]:
def predict_decision_tree(row):
    # Nodo radice: BMI_Category
    if row["BMI_Category"] == "Obese":
        return 1                      # foglia pura -> diabetico

    elif row["BMI_Category"] == "Overweight":
        # split su GenHlth: {2,4} -> diabetico, {1,3} -> non diabetico
        if row["GenHlth"] in [2, 4]:
            return 1
        else:
            return 0

    else:  # BMI_Category == "Normal"
        # split su HighBP: entrambi i rami portano a classe 0
        return 0                      # Normal -> non diabetico

### Verifica su un singolo campione

Si verifica il funzionamento della funzione sul primo campione del dataset
(BMI=21 → Normal, classe reale = 0).

In [16]:
# Test sul primo campione

primo = manual_df.iloc[0]
pred = predict_decision_tree(primo)

print("BMI_Category:", primo["BMI_Category"])
print("GenHlth:", primo["GenHlth"], "| HighBP:", primo["HighBP"])
print("Predizione:", pred)
print("Classe reale:", primo["Diabetes_binary"])

BMI_Category: Normal
GenHlth: 3 | HighBP: 1
Predizione: 0
Classe reale: 0


Il primo campione (BMI Normal) viene correttamente classificato come **0**, in accordo con
la regola 4 dell'albero.

## 12. Predizione su tutti i campioni del dataset manuale

Si applica il classificatore a tutte le 14 osservazioni e si confronta la classe predetta
con quella reale.

In [17]:
# Predizione su tutti i campioni

manual_df["Predicted"] = manual_df.apply(predict_decision_tree, axis=1)

manual_df[["BMI_Category", "GenHlth", "HighBP", "Diabetes_binary", "Predicted"]]

,BMI_Category,GenHlth,HighBP,Diabetes_binary,Predicted
0,Normal,3,1,0,0
1,Overweight,2,0,0,1
2,Obese,3,1,1,1
3,Normal,3,0,0,0
4,Obese,4,0,1,1
5,Overweight,3,1,0,0
6,Overweight,4,1,1,1
7,Overweight,2,0,1,1
8,Normal,1,1,0,0
9,Overweight,2,1,1,1


### Analisi degli errori

Confrontando le colonne `Diabetes_binary` e `Predicted` si individuano i seguenti errori:

- **Riga 1** (Overweight, GenHlth=2): predetto 1, reale 0 → **falso positivo (FP)**
- **Riga 12** (Normal, GenHlth=3): predetto 0, reale 1 → **falso negativo (FN)**

In totale **12 campioni su 14** vengono classificati correttamente: un errore in meno rispetto
alla versione dell'albero basata su HighChol.

## 13. Valutazione delle prestazioni

Si calcolano le stesse metriche usate per il Naive Bayes, in modo da poter confrontare
direttamente i due classificatori:

- Accuracy
- Confusion Matrix
- Precision
- Recall
- F1-Score

### Accuracy

L'accuracy rappresenta la percentuale di osservazioni classificate correttamente:

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

In [18]:
accuracy = (manual_df["Diabetes_binary"] == manual_df["Predicted"]).mean()

print("Accuracy:", accuracy)

Accuracy:

 0.8571428571428571


### Confusion Matrix

La matrice di confusione permette di analizzare nel dettaglio la tipologia degli errori.

In [19]:
tp = len(manual_df[(manual_df["Diabetes_binary"] == 1) & (manual_df["Predicted"] == 1)])
tn = len(manual_df[(manual_df["Diabetes_binary"] == 0) & (manual_df["Predicted"] == 0)])
fp = len(manual_df[(manual_df["Diabetes_binary"] == 0) & (manual_df["Predicted"] == 1)])
fn = len(manual_df[(manual_df["Diabetes_binary"] == 1) & (manual_df["Predicted"] == 0)])

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

# Visualizzazione della matrice
confusion_matrix_df = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)

confusion_matrix_df

TP = 6
TN = 6
FP = 1
FN = 1


,Predicted 0,Predicted 1
Actual 0,6,1
Actual 1,1,6


### Precision

La precision misura quanti dei campioni predetti come positivi sono effettivamente positivi:

$$Precision = \frac{TP}{TP + FP}$$

Una precision elevata indica che il modello produce pochi falsi positivi.

In [20]:
precision = tp / (tp + fp)

print("Precision:", precision)

Precision: 0.8571428571428571


### Recall

La recall misura la capacità del classificatore di individuare i campioni positivi:

$$Recall = \frac{TP}{TP + FN}$$

Una recall elevata indica che il modello perde pochi casi positivi, ovvero produce
pochi falsi negativi.

In [21]:
recall = tp / (tp + fn)

print("Recall:", recall)

Recall: 0.8571428571428571


### F1-Score

L'F1-Score è la media armonica tra precision e recall:

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

È utile quando si cerca un compromesso tra precision e recall.

In [22]:
f1 = 2 * (precision * recall) / (precision + recall)

print("F1 Score:", f1)

F1 Score: 0.8571428571428571


In [23]:
# Tabella risultati

print("========== RISULTATI ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

========== RISULTATI ==========
Accuracy : 0.8571
Precision: 0.8571
Recall   : 0.8571
F1 Score : 0.8571


## 14. Discussione critica dei risultati

Il classificatore **Decision Tree** implementato manualmente ha ottenuto i seguenti risultati
sul dataset `manuale.csv`:

- Accuracy = 85.71%
- Precision = 85.71%
- Recall = 85.71%
- F1-Score = 85.71%

La Confusion Matrix ottenuta è:

|                | Predicted 0 | Predicted 1 |
|----------------|------------|------------|
| Actual 0       | 6          | 1          |
| Actual 1       | 1          | 6          |

(TN = 6, FP = 1, FN = 1, TP = 6)

### Ruolo della scelta dello split (Information Gain)

Un aspetto metodologico rilevante riguarda la scelta della feature per il ramo Overweight.
Adottando **GenHlth** (la feature con Information Gain massimo su quel sotto-insieme, ≈ 0.541)
invece di una feature con guadagno inferiore, il classificatore migliora le proprie prestazioni:
l'accuracy passa dall'78.57% all'85.71% e il numero di errori si riduce da 3 a 2.

Questo conferma sperimentalmente il principio teorico alla base degli alberi di decisione:
**scegliere ad ogni nodo la feature che massimizza l'Information Gain** porta, in generale, a
partizioni più pure e quindi a un classificatore più accurato. Tutte e quattro le metriche
risultano bilanciate sullo stesso valore (85.71%), segno che il modello non privilegia una
classe a scapito dell'altra.

### Confronto con il Naive Bayes

Rispetto al classificatore Naive Bayes (che otteneva 78.57% di accuracy e 76.92% di F1-Score),
questo Decision Tree risulta più performante sul dataset manuale. È importante però sottolineare
che, con soli 14 campioni, queste differenze non sono statisticamente significative: il confronto
realmente affidabile tra i due modelli avverrà nei Task 4 e 5, su `training.csv`.

I due errori residui corrispondono a campioni "atipici":
- un soggetto Overweight con buona salute percepita (GenHlth=2) ma non diabetico (riga 1);
- un soggetto con BMI normale ma diabetico (riga 12).

Questi casi contraddicono il pattern generale e risultano difficili da classificare per qualunque modello.

### Vantaggi e limiti del Decision Tree

**Vantaggi:**
- produce **regole esplicite e facilmente interpretabili**, leggibili anche da un non esperto;
- non richiede assunzioni di indipendenza tra le feature (a differenza del Naive Bayes);
- gestisce naturalmente le interazioni tra feature (es. BMI *combinato con* stato di salute).

**Limiti:**
- su soli 14 campioni l'albero è molto sensibile ai singoli dati e tende a "memorizzare" i casi
  piuttosto che a generalizzare (rischio di **overfitting**, evidenziato dalle foglie con un solo campione);
- la scelta degli split e la struttura dell'albero sono fortemente influenzate dalla ridotta numerosità
  del campione.

### Conclusione

Le prestazioni sono soddisfacenti e superiori a quelle del Naive Bayes sul dataset manuale.
La scelta di seguire rigorosamente l'Information Gain (GenHlth nel ramo Overweight) si è rivelata
vincente. Nei Task 4 e 5 entrambi i classificatori manuali verranno valutati su `training.csv`
(50.000 campioni), permettendo una stima più affidabile e un confronto solido con i classificatori
addestrati tramite Scikit-Learn.